In [2]:
import pandas as pd
import numpy as np
import pickle
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import train_test_split

print("1. A carregar, limpar e balancear os dados...")
df = pd.read_csv('dataset_final_teste.csv', sep=';')
df = df[df['Label'] != 'Label']
df['Text'] = df['Text'].str.replace(r'<ref.*?>.*?</ref>|<ref.*?>', '', regex=True)

min_count = df['Label'].value_counts().min()
df_balanced = df.groupby('Label').sample(n=min_count, random_state=42)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

textos = df_balanced['Text'].values
labels = df_balanced['Label'].values

PROFESSOR_MAP = {'Human': 0, 'OpenAI': 1, 'Google': 2, 'Meta': 3, 'Anthropic': 4}
y = np.array([PROFESSOR_MAP[label] for label in labels])

# --- 1. SEPARAR O TEXTO CRU PRIMEIRO (Evita Data Leakage) ---
X_train_full_text, X_test_text, y_train_full, y_test = train_test_split(
    textos, y, test_size=0.10, random_state=42, stratify=y
)

# --- 2. CONFIGURAR O VETORIZADOR (Palavras + Caracteres) ---
# Usamos menos features para focar nos padrões reais e não memorizar o dataset
char_vect = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=1500)
word_vect = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=1000)

vectorizer = FeatureUnion([
    ("char", char_vect),
    ("word", word_vect)
])

print("\n2. A executar TF-IDF (APENAS no Treino para aprender vocabulário)...")
# O vectorizer APRENDE as regras SÓ com os dados de treino
vectorizer.fit(X_train_full_text)

# --- 3. GUARDAR O VETORIZADOR CORRETO PARA A SUBMISSÃO ---
os.makedirs('Subm3', exist_ok=True)
with open('Subm3/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# --- 4. TRANSFORMAR OS TEXTOS EM MATRIZES ---
X_train_full = vectorizer.transform(X_train_full_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()

# --- 5. GUARDAR FICHEIROS NPY ---
np.save('X_train_full.npy', X_train_full)
np.save('y_train_full.npy', y_train_full)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)

print(f"\nSucesso! Dados particionados e Vetorizador alinhado:")
print(f"  -> X_train_full (para K-Fold):  {X_train_full.shape}")
print(f"  -> X_test (avaliação final): {X_test.shape}")

1. A carregar, limpar e balancear os dados...

2. A executar TF-IDF (APENAS no Treino para aprender vocabulário)...

Sucesso! Dados particionados e Vetorizador alinhado:
  -> X_train_full (para K-Fold):  (4054, 2500)
  -> X_test (avaliação final): (451, 2500)
